# Аналитический аудит и выявление рисков

**Резюме результатов анализа модели «КадроБот 3000»**

## 🔍 Технический анализ: идентификация источников дискриминации

### Статистически значимые различия в оценках модели


| Возрастная группа | Доля приглашений (model_decision) | Объективная успешность (true_success) | Исторический найм (historical_success) |
| :--- | :--- | :--- | :--- |
| <30 | 94.1% | 41.4% | 74.1% |
| 30-45 | 46.8% | 37.4% | 49.2% |
| 45+ | 3.8% | 35.9% | 21.3% |


**Вывод**: Различия в оценках модели между возрастными группами статистически значимы (тест Крускала-Уоллиса: H=3377.62, p≈0.00).

### Прокси-переменные для возраста


| Признак | Корреляция с возрастом | Корреляция с model\_score | Механизм дискриминации |
| :--- | :--- | :--- | :--- |
| `graduation_year` | -0.884 | +0.878 | Более ранний год окончания вуза → пониженный скор |
| `outdated_vocab` | +0.526 | -0.699 | Наличие «устаревшей» лексики → пониженный скор |
| `youth_hobby` | -0.485 | +0.576 | Отсутствие «молодёжных» хобби → пониженный скор |
| `experience` | +0.913 | -0.805 | Большой опыт → парадоксально низкий скор |



**Ключевой вывод**: Модель не использует `age` напрямую, но воспроизводит возрастную дискриминацию через коррелирующие признаки, унаследованные из исторических данных.

### Условный анализ (контроль образования и опыта)


При фиксированных значениях `education` и `experience` средняя разница в `model_score` между группами $45+$ и $<30$ составляет $-0.316$.

Это подтверждает, что дискриминация не объясняется различиями в квалификации — модель систематически занижает оценки кандидатам $45+$ даже при равных профессиональных характеристиках.

## ⚖️ Этический анализ: оценка последствий и несправедливости

### Нарушение принципов справедливости



| Принцип справедливости | Математическая формула | Значение | Уточненная бизнес-интерпретация (на тестовых данных) |
| :--- | :--- | :--- | :--- |
| **Демографический паритет** | $\vert P(\hat{Y}=1 \vert A = \text{<30}) - P(\hat{Y}=1 \vert A = \text{45+}) \vert$ | **0.903** | **Критический перекос воронки:** Независимо от реальной квалификации, кандидат из молодой группы имеет в **24.8×** более высокий шанс получить приглашение на первичный этап. |
| **Равные возможности** *(Equal Opportunity)* | $\vert TPR_{\text{<30}} - TPR_{\text{45+}} \vert$ | **0.890** | **Дискриминация талантов:** Модель успешно одобряет $94.7\%$ продуктивных молодых специалистов, но пропускает лишь $5.7\%$ аналогично успешных кандидатов старше 45 лет (разрыв в **16.6×**). |


### Кто страдает от дискриминации?


* **Кандидаты 45+**: Систематически исключаются из процесса найма несмотря на сопоставимую объективную успешность.
* **Работодатель («МаркетПлюс»)**:
* * Потеря квалифицированных кадров из-за ложных отказов (False Negatives);
* * Репутационные риски при публичном скандале (прецедент Amazon, 2018);
* * Юридические риски: нарушение ст. 3 ТК РФ (запрет дискриминации по возрасту).
* **Общество**: Усиление возрастной сегрегации на рынке труда, эрозия принципа равных возможностей.

### Философская оценка


| Этическая теория | Оценка ситуации |
| :--- | :--- |
| **Утилитаризм** | Модель снижает общую полезность: ложные отказы талантливым кандидатам 45+ уменьшают эффективность найма и социальное благополучие |
| **Либертарианство** | Нарушается свобода договора и меритократия: алгоритм судит человека по групповому признаку (возрасту) вместо оценки его индивидуальных навыков, ограничивая право на честный рыночный обмен компетенциями |
| **Деонтология** | Нарушается категорический императив: кандидаты 45+ используются как средство для оптимизации метрик, а не как цель сами по себе |
| **Теория справедливости Ролза** | Нарушается «принцип различия»: неравенство не работает в пользу наименее преуспевающих (кандидатов 45+) |
| **Этика добродетели** (Virtue Ethics) | Компания теряет честность и справедливость: автоматизация предвзятости противоречит формированию характера благодетельной организации |

## ⚖️ Правовой анализ: соответствие нормативным требованиям


### Нарушения российского законодательства


| Норма | Содержание | Применимость к кейсу |
| :--- | :--- | :--- |
| **Ст. 3 ТК РФ** | Запрет дискриминации в сфере труда, в т.ч. по возрасту | Модель систематически ограничивает права кандидатов 45+ |
| **152-ФЗ «О персональных данных»** | Требования к обработке ПДн, включая автоматизированные решения | Отсутствие прозрачности в логике принятия решений моделью |
| **Ст. 10 ГК РФ** | Запрет злоупотребления правом | Использование исторически предвзятых данных без коррекции |


### Международные стандарты



* **GDPR (ст. 22)**: Право субъекта не подвергаться решению, основанному исключительно на автоматизированной обработке, если оно порождает юридические последствия.
* **UNESCO Recommendation on AI Ethics (2021)**: Принцип справедливости и недискриминации, требование оценки воздействия на права человека.

## 🚨 Итоговая оценка рисков



| Тип риска | Уровень | Обоснование |
| :--- | :--- | :--- |
| **Репутационный** | 🔴 Высокий | Публичная дискриминация по возрасту → медиа-скандал, потеря доверия |
| **Юридический** | 🔴 Высокий | Прямое нарушение ст. 3 ТК РФ; риск коллективных исков |
| **Бизнес-риск** | 🟡 Средний | Потеря квалифицированных кадров; снижение качества найма |
| **Этический** | 🔴 Высокий | Систематическое нарушение принципов справедливости и прав человека |
| **Технический** | 🟡 Средний | Модель воспроизводит исторические смещения; низкая обобщающая способность |
